In [ ]:
from utils import create_candidate_regions, create_bam_index
import pyranges1 as pr
import pandas as pd
import subprocess
from pathlib import Path
import argparse

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
gene_sets_filepath = 'data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz'

gene_sets_transcripts_filepath = gene_sets_filepath.replace('.gtf', '.transcripts.gtf')
gene_sets_cds_filepath = gene_sets_filepath.replace('.gtf', '.cds.gtf')

command = f"""gunzip -c {gene_sets_filepath} | awk -F '\t' '$3 == "transcript"' | gzip > {gene_sets_transcripts_filepath}"""
if not Path(gene_sets_transcripts_filepath).is_file():
    subprocess.run(command, shell=True, executable="/bin/bash", check=True)

command = f"""gunzip -c {gene_sets_filepath} | awk -F '\t' '$3 =="CDS" || $3 == "stop_codon"' | gzip > {gene_sets_cds_filepath}"""
if not Path(gene_sets_cds_filepath).is_file():
    subprocess.run(command, shell=True, executable="/bin/bash", check=True)

In [4]:
# Перевести названия хромосом из формата Ensembl в формат UCSC
ensembl_ucsc_chr_names_translation = True

In [5]:
# ==============================================================================================================
# Оставляем только 1 транскрипт на каждый ген (gene_id).
# Исключением являются гены для которых есть MANE_Plus_Clinical, у них может быть два транскрипта.
# ==============================================================================================================

# --------------------------------------------------------------------------------------------------------------
# Подгружаем pr содержащий только транскрипты.
# (Можно загрузить весь и фильтрануть: Feature == "transcript")
# --------------------------------------------------------------------------------------------------------------
gtf_pr_transcripts = pr.read_gtf(gene_sets_transcripts_filepath, duplicate_attr=True)

# --------------------------------------------------------------------------------------------------------------
# Перевод названия хромосом из формата Ensembl в формат UCSC (при необходимости).
# --------------------------------------------------------------------------------------------------------------
if ensembl_ucsc_chr_names_translation:
    gtf_pr_transcripts['Chromosome'] = ('chr' + gtf_pr_transcripts['Chromosome'].astype(str)).replace({'chrMT': 'chrM'})

# --------------------------------------------------------------------------------------------------------------
# Превращение тэгов в список и разбиение на разные строки (взрыв/explode).
# --------------------------------------------------------------------------------------------------------------
gtf_pr_transcripts['tag'] = gtf_pr_transcripts['tag'].str.split(',')
gtf_pr_transcripts_exploded = gtf_pr_transcripts.explode('tag')

# --------------------------------------------------------------------------------------------------------------
# Задаем и присваиваем приоритет каждому транскрипту по тэгу и биотипу транскриптов для дальнейшей сортировки с их участием
# --------------------------------------------------------------------------------------------------------------
tag_priority = {
    'MANE_Select': 0,
    'Ensembl_canonical': 1,
    'gencode_primary': 2,
    'gencode_basic': 3,
    'CCDS': 4,
}

transcript_biotype_priority = {
    'protein_coding': 0,
}

gtf_pr_transcripts_exploded['tag_priority'] = gtf_pr_transcripts_exploded['tag'].map(tag_priority).astype('Int64')
gtf_pr_transcripts_exploded['transcript_biotype_priority'] = gtf_pr_transcripts_exploded['tag'].map(transcript_biotype_priority).astype('Int64')

# --------------------------------------------------------------------------------------------------------------
# Сортируем
# --------------------------------------------------------------------------------------------------------------
gtf_pr_transcripts_exploded = gtf_pr_transcripts_exploded.sort_values(['gene_id', 'tag_priority', 'transcript_support_level', 'transcript_biotype_priority'], na_position='last')

# --------------------------------------------------------------------------------------------------------------
# Оставляем только первый транскрипт для каждого гена (gene_id)
# --------------------------------------------------------------------------------------------------------------
gtf_pr_transcripts_singe_transcript = gtf_pr_transcripts_exploded.drop_duplicates(subset=['gene_id'], keep='first')

# --------------------------------------------------------------------------------------------------------------
# Собираем MANE_Plus_Clinical транскрипты
# --------------------------------------------------------------------------------------------------------------
gtf_pr_transcripts_mane_plus_clinical = gtf_pr_transcripts_exploded.query('tag == "MANE_Plus_Clinical"')

# --------------------------------------------------------------------------------------------------------------
# Объединяем датафреймы с основными транскриптами и транскриптами MANE_Plus_Clinical.
# Убираем дубликаты на тот случай если MANE_Plus_Clinical уже является основным.
# --------------------------------------------------------------------------------------------------------------
gtf_pr_main_transcripts = pd.concat([gtf_pr_transcripts_singe_transcript, gtf_pr_transcripts_mane_plus_clinical], ignore_index=True)
gtf_pr_main_transcripts = gtf_pr_main_transcripts.drop_duplicates()
gtf_pr_main_transcripts = gtf_pr_main_transcripts.drop(['tag_priority', 'transcript_biotype_priority'], axis=1)

In [6]:
# ==============================================================================================================
# Оставляем CDS и stop_codon только по одному основному транскрипту (+ MANE_Plus_Clinical).
# ==============================================================================================================
gtf_pr_cds = pr.read_gtf(gene_sets_cds_filepath)

# --------------------------------------------------------------------------------------------------------------
# Перевод названия хромосом из формата Ensembl в формат UCSC (при необходимости).
# --------------------------------------------------------------------------------------------------------------
if ensembl_ucsc_chr_names_translation:
    gtf_pr_cds['Chromosome'] = ('chr' + gtf_pr_cds['Chromosome'].astype(str)).replace({'chrMT': 'chrM'})

# --------------------------------------------------------------------------------------------------------------
# Оставляем только по одному основному транскрипту для каждого гена (+ MANE_Plus_Clinical).
# --------------------------------------------------------------------------------------------------------------
gtf_pr_cds_single_transcript = gtf_pr_cds[gtf_pr_cds['transcript_id'].isin(gtf_pr_main_transcripts['transcript_id'])]

# --------------------------------------------------------------------------------------------------------------
# Оставляем только основные столбцы для BED.
# --------------------------------------------------------------------------------------------------------------
gtf_pr_cds_single_transcript_short = gtf_pr_cds_single_transcript[['Chromosome', 'Start', 'End']]
gtf_pr_cds_single_transcript_overlapped = (gtf_pr_cds_single_transcript_short
                                           .merge_overlaps()
                                           .sort_ranges())

# --------------------------------------------------------------------------------------------------------------
# Сохраняем BED.
# --------------------------------------------------------------------------------------------------------------
cds_bed_filepath = 'data/refs/cds_overlapped.bed'
gtf_pr_cds_single_transcript_overlapped.to_bed(cds_bed_filepath)

In [ ]:
parser = argparse.ArgumentParser()

parser.add_argument('--input_vcf', '-i', dest='input_vcf', required=True, help='Path to input VCF')
parser.add_argument('--fasta', '-f', dest='fasta', required=True, help='Path to input FASTA')
parser.add_argument('--bam', '-b', dest='bam', required=True, help='Path to input BAM')
parser.add_argument('--genozip_ref', '-gr', dest='genozip_ref', help='Path to input genozip reference')
parser.add_argument('--bed', help='Path to BED')
parser.add_argument('--mnv_size', '-s', dest='mnv_size', type=int, default=3, help='Max region size of MNV in inner BED')
parser.add_argument('--output_vcf', '-o', dest='output_vcf', required=True, help='Path to output VCF')
parser.add_argument('--compress', '-c', dest='compression', default=1, type=int, help='Compress output VCF with bgzip')
parser.add_argument('--remove_intermediate_files', '-rm', dest='rm_intermediate_files', default=0, type=int, help='Remove intermediate files')

args = parser.parse_args()

if args.bed and not args.input_vcf.endswith('.gz'):
    raise ValueError("Using BED requires a BGZF-compressed and indexed VCF")

# ==============================================================================================================
# Индексируем VCF (если сжат BGZIP)
# ==============================================================================================================
if args.input_vcf.endswith('.gz'):
    command = f'bcftools index -f -t "{args.input_vcf}"'
    subprocess.run(command, shell=True, check=True)

# ==============================================================================================================
# Создаем BED файл, который включает в себя регионы с интервалами позиций вариантов из vcf файла, находящихся на расстоянии не более заданного (по дефотку = 3 (расстояние одного триплета)).
# ==============================================================================================================
mnv_candidates_pr = create_candidate_regions(args.input_vcf, args.bed, distance=args.mnv_size) # Даже если BED не передан, то args.bed вернет None, что функция учитывает.
mnv_candidates_bed_filepath = args.input_vcf.removesuffix('.gz').removesuffix('.vcf') + '.mnv_candidates.bed'
mnv_candidates_pr.to_bed(mnv_candidates_bed_filepath)

# ==============================================================================================================
# Если используется Genozip BAM и BED файл, то нужно извлечь регионы в новый BAM.
# ==============================================================================================================
if args.bam.endswith('.bam.genozip'):
    if not args.genozip_ref:
        raise ValueError('Genozip BAM requires genozip reference')
    
    # --------------------------------------------------------------------------------------------------------------
    # Создание BED (TSV) файла с расширенными интервалами для извлечения ридов из .bam.genozip
    # --------------------------------------------------------------------------------------------------------------
    mnv_candidates_pr_extended = (
        mnv_candidates_pr.extend_ranges(ext_5=170, use_strand=False) # Делаем отступы влево на длину рида (или больше) из-за особенностей извлечения ридов genozip.
        .clip_ranges() # Нужен чтобы при отступе влево не уйти в отрицательные позиции хромосомы.
        .merge_overlaps()
        .sort_ranges()
    )

    # Нужно учитывать, что BED (TSV) это 0-based формат, а genozip использует 1-based координаты.
    # Таким образом, без дополнительной корректировки (которая здесь делаться не будет), мы захватим 1 дополнительную полизицию слева.
    mnv_candidates_extended_filepath = mnv_candidates_bed_filepath.removesuffix('.bed') + '_extended.tsv'
    mnv_candidates_pr_extended.to_csv(mnv_candidates_extended_filepath, index=False, header=False, sep='\t')

    # --------------------------------------------------------------------------------------------------------------
    # Извлекает риды из .bam.genozip по расширенным регионам mnv_candidates, сохраняем в BAM.
    # --------------------------------------------------------------------------------------------------------------
    bam_filepath = args.bam.removesuffix('.genozip').removesuffix('.bam') + '.extracted.bam'
    intermediate_bam_filepath = bam_filepath # Для последующего удаления, не затрагивая оригинальный BAM

    command = f'''genocat "{args.bam}" \
        --reference "{args.genozip_ref}" \
        -R "{mnv_candidates_extended_filepath}" \
        --force \
        --output "{bam_filepath}"'''
    subprocess.run(command, shell=True, check=True)

    # --------------------------------------------------------------------------------------------------------------
    # Индексируем новый BAM
    # --------------------------------------------------------------------------------------------------------------
    command = f'samtools index "{bam_filepath}"'
    subprocess.run(command, shell=True, check=True)
else:
    bam_filepath = args.bam
    create_bam_index(bam_filepath)

# ==============================================================================================================
# Вариант коллинг при помощи FreeBayes.
# ==============================================================================================================

# --------------------------------------------------------------------------------------------------------------
# Запускаем коллинг
# --------------------------------------------------------------------------------------------------------------
compression = '| bgzip -c' if args.compression == 1 else ''

# --haplotype-length несовсем подходящее название. Его другое имя — --max-complex-gap 3, то есть «максимальный промежуток совпадений с референсом между изменениями».
command = f'''freebayes \
        -f "{args.fasta}" \
        -t "{mnv_candidates_bed_filepath}" \
        --haplotype-length 1 \
        "{bam_filepath}" \
        | bcftools norm -m -any \
        | vcffilter -f "TYPE = complex | TYPE = mnp" \
        {compression} > "{args.output_vcf}"'''
subprocess.run(command, shell=True, executable="/bin/bash", check=True)

if args.rm_intermediate_files:
    Path(mnv_candidates_bed_filepath).unlink(missing_ok=True)
    if args.bam.endswith('.bam.genozip'):
        Path(mnv_candidates_extended_filepath).unlink(missing_ok=True)
        Path(intermediate_bam_filepath).unlink(missing_ok=True)
        Path(intermediate_bam_filepath + '.bai').unlink(missing_ok=True)

In [7]:
# ==============================================================================================================
# Создаем BED файл, который включает в себя регионы с интервалами позиций вариантов из vcf файла, находящихся на расстоянии не более заданного (по дефотку = 3 (расстояние одного триплета)).
# ==============================================================================================================
input_vcf_filepath = 'data/test_files/vcf/000007239270_DeepVariant.vcf.gz'
mnv_candidates_bed_filepath = input_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.mnv_candidates.bed'
mnv_candidates_pr = create_candidate_regions(input_vcf_filepath, cds_bed_filepath)
mnv_candidates_pr.to_bed(mnv_candidates_bed_filepath)


In [8]:
# ==============================================================================================================
# Создание BED (TSV) файла с расширенными интервалами для извлечения ридов из .bam.genozip
# ==============================================================================================================
mnv_candidates_pr_extended = (
    mnv_candidates_pr.extend_ranges(ext_5=170, use_strand=False) # Делаем отступы влево на длину рида (или больше) из-за особенностей извлечения ридов genozip.
    .clip_ranges() # Нужен чтобы при отступе влево не уйти в отрицательные позиции хромосомы.
    .merge_overlaps()
    .sort_ranges()
)

# Нужно учитывать, что BED (TSV) это 0-based формат, а genozip использует 1-based координаты.
# Таким образом, без дополнительной корректировки (которая здесь делаться не будет), мы захватим 1 дополнительную полизицию слева.
mnv_candidates_extended_filepath = 'data/refs/mnv_candidates_extended.tsv'
mnv_candidates_pr_extended.to_csv(mnv_candidates_extended_filepath, index=False, header=False, sep='\t')

In [9]:
# ==============================================================================================================
# Извлекает риды из .bam.genozip по расширенным регионам mnv_candidates, сохраняем в bam
# ==============================================================================================================
bam_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.bam.genozip'
reference_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.ref.genozip'
regions_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/mnv_candidates_extended.tsv'
bam_output_filepath = bam_genozip_filepath.removesuffix('.genozip').removesuffix('.bam') + '.extracted.bam'

command = f'''genocat {bam_genozip_filepath} \
    --reference {reference_genozip_filepath} \
    -R {regions_filepath} \
    --force \
    --output {bam_output_filepath}'''
subprocess.run(command, shell=True, executable="/bin/bash", check=True)

# --------------------------------------------------------------------------------------------------------------
# Индексируем новый BAM
# --------------------------------------------------------------------------------------------------------------
command = f'samtools index {bam_output_filepath}'
subprocess.run(command, shell=True, executable="/bin/bash", check=True)

genocat /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.extracted.bam : 0% (0 second            0% (19 minutes 27 second                        0% (17 minutes 50 second                        0% (17 minutes 20 second                        0% (17 minutes 5 seconds                        0% (16 minutes 56 second                        0% (16 minutes 50 second                        0% (16 minutes 45 second                        0% (16 minutes 41 second                        Done (8 seconds)


CompletedProcess(args='samtools index /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.extracted.bam', returncode=0)

In [18]:
# ==============================================================================================================
# Вариант коллинг на основе имеющегося vcf файла при помощи FreeBayes в регионах BED файла. 
# По сути FreeBayes берет имеющиеся варианты и заново их оценивает превращая в MNV в случае cis-положения.
# ==============================================================================================================
fasta_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa'
mnv_candidates_vcf_filepath = input_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.mnv_candidates.vcf.gz'

# --------------------------------------------------------------------------------------------------------------
# Индексируем VCF (если индекс новее .vcf, то не обязательно)
# --------------------------------------------------------------------------------------------------------------
command = f'bcftools index -f -t {input_vcf_filepath}'
subprocess.run(command, shell=True, executable="/bin/bash", check=True)

# --------------------------------------------------------------------------------------------------------------
# Запускаем коллинг
# --------------------------------------------------------------------------------------------------------------

# --haplotype-length несовсем подходящее название. Его другое имя — --max-complex-gap 3, то есть «максимальный промежуток совпадений с референсом между изменениями».
command = f'''freebayes \
    -f {fasta_filepath} \
    -t {mnv_candidates_bed_filepath} \
    --haplotype-length 1 \
    {bam_output_filepath} \
    | bcftools norm -m -any \
    | vcffilter -f "TYPE = complex | TYPE = mnp" \
    | bgzip -c > {mnv_candidates_vcf_filepath}'''
subprocess.run(command, shell=True, executable="/bin/bash", check=True)

Lines   total/split/joined/realigned/removed/skipped:	320/42/0/0/0/0


CompletedProcess(args='freebayes     -f /Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa     -t data/test_files/vcf/000007239270_DeepVariant.mnv_candidates.bed     --haplotype-length 1     /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.extracted.bam     | bcftools norm -m -any     | vcffilter -f "TYPE = complex | TYPE = mnp"     | bgzip -c > data/test_files/vcf/000007239270_DeepVariant.mnv_candidates.vcf.gz', returncode=0)